# Pseudo-Label v2 + Submission Export Pipeline

**Repository**: Joyoem/ArgFan  
**Task**: UZH Shared Task – ArgMining 2026 (UN-Resolution paragraph annotation)  
**Model**: Qwen2.5-7B (4-bit NF4, local path)  

## What this notebook does
1. Loads `coreset.csv`, optional `human_annotation.json` few-shots, and a tag-whitelist CSV.
2. Runs a **v2 prompt** (English-only `think`, structured JSON output) through a locally loaded Qwen model with deterministic generation (`do_sample=False`).
3. Applies rule-based post-processors:
   - **Trigger extractor** – detects French preambular/operative trigger words.
   - **Boundary checker** – flags likely mis-split or incomplete paragraphs.
   - **Constraint checker** – enforces tag-whitelist, valid relation types, pid-in-context, and Top-K link cap.
4. Resumes from an existing checkpoint by `(doc_id, pid)` so interrupted runs continue cleanly.
5. Saves:
   - `pseudo_labels_v2.jsonl` – raw model output + post-processing fields
   - `pseudo_labels_v2_clean.jsonl` – submission-ready fields only
   - `review_queue_v2.jsonl` – rows flagged for human review
   - `quality_report_v2.json` – aggregate statistics
   - `submission_export/<TEXT_ID>.json` – one file per document in competition schema

## V0 → V2 fixes
| Issue | Fix |
|-------|-----|
| Variable `MAX_NEW` / `CTX_N` inconsistency | Renamed to `MAX_NEW_TOKENS` / `CTX_WINDOW` everywhere |
| Conflicting Chinese `think` example in user prompt | Removed; all `think` strings are short templated English |
| Stochastic generation (`temperature`, `top_p`, `top_k`) | `do_sample=False`; those kwargs are never passed |
| No resume – restarted from row 0 | Resume by `(doc_id, pid)` set built from existing output file |

In [ ]:
# Install / upgrade required packages.
# Run once per environment; comment out if already installed.
import subprocess, sys
pkgs = [
    "transformers>=4.40.0",
    "accelerate>=0.27.0",
    "bitsandbytes>=0.43.0",
    "pandas>=2.0",
    "tqdm",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)

In [ ]:
import json
import os
import re
import pathlib
import datetime
import collections
from typing import Optional

import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# ─── USER-CONFIGURABLE ────────────────────────────────────────────────────────
# Paths
BASE_DIR    = "/path/to/your/data"           # folder containing coreset.csv etc.
MODEL_PATH  = "/path/to/Qwen2.5-7B"         # local Qwen2.5-7B weights

# Which tag-whitelist CSV to use:
#   education_dimensions_updated.csv  → train split  (use during development / offline eval)
#   evaluation_dimensions_updated.csv → test/blind   (use when preparing the final submission)
# Switch to education_dimensions_updated.csv for iterative development;
# switch back to evaluation_dimensions_updated.csv before submitting.
WHITELIST_CSV = os.path.join(BASE_DIR, "evaluation_dimensions_updated.csv")

# Optional: path to human_annotation.json for few-shot prompting (None = skip)
FEWSHOT_JSON  = os.path.join(BASE_DIR, "human_annotation.json")

# Output directories
OUTPUT_DIR     = os.path.join(BASE_DIR, "outputs_v2")
SUBMISSION_DIR = os.path.join(OUTPUT_DIR, "submission_export")

# ─── INFERENCE KNOBS ─────────────────────────────────────────────────────────
MAX_NEW_TOKENS = 512   # max tokens generated per paragraph
CTX_WINDOW     = 4096  # model context window (truncate context if needed)
TOP_K_LINKS    = 5     # max entries in matched_paras per paragraph
BATCH_SIZE     = 1     # keep at 1 (each para is independent; simplifies resumption)

# ─── INIT OUTPUT DIRS ─────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR,     exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

print(f"BASE_DIR      : {BASE_DIR}")
print(f"MODEL_PATH    : {MODEL_PATH}")
print(f"WHITELIST_CSV : {WHITELIST_CSV}")
print(f"OUTPUT_DIR    : {OUTPUT_DIR}")
print(f"SUBMISSION_DIR: {SUBMISSION_DIR}")

In [ ]:
def load_whitelist(csv_path: str) -> list:
    """Load tag labels from a whitelist CSV.

    Looks for a column named 'dimension'; falls back to the first column.
    Returns a sorted, deduplicated list of non-empty string labels.
    """
    df = pd.read_csv(csv_path)
    col = "dimension" if "dimension" in df.columns else df.columns[0]
    labels = sorted({
        lbl.strip()
        for lbl in df[col].dropna().tolist()
        if str(lbl).strip()
    })
    print(f"Loaded {len(labels)} whitelist labels from: {csv_path}")
    return labels


if os.path.exists(WHITELIST_CSV):
    WHITELIST = load_whitelist(WHITELIST_CSV)
else:
    print(f"WARNING: WHITELIST_CSV not found ({WHITELIST_CSV}). Using empty whitelist.")
    WHITELIST = []

print("First 10 labels:", WHITELIST[:10])

In [ ]:
def load_coreset(csv_path: str) -> pd.DataFrame:
    """Load coreset.csv.

    Required columns: doc_id, pid, text
    Optional columns: any others are preserved and attached to each row.
    """
    df = pd.read_csv(csv_path)
    for col in ("doc_id", "pid", "text"):
        if col not in df.columns:
            raise ValueError(f"coreset.csv is missing required column: '{col}'")
    df["pid"] = df["pid"].astype(int)
    df["doc_id"] = df["doc_id"].astype(str).str.strip()
    total = len(df)
    docs  = df["doc_id"].nunique()
    print(f"Loaded coreset: {total} rows across {docs} documents")
    return df


CORESET_PATH = os.path.join(BASE_DIR, "coreset.csv")
if os.path.exists(CORESET_PATH):
    coreset_df = load_coreset(CORESET_PATH)
    coreset_df.head(3)
else:
    print(f"WARNING: coreset.csv not found at {CORESET_PATH}. Using empty DataFrame for dry-run.")
    coreset_df = pd.DataFrame(columns=["doc_id", "pid", "text"])

In [ ]:
def load_fewshot_examples(json_path: Optional[str]) -> list:
    """Load human annotation examples for few-shot prompting.

    Expected format: a list of dicts, each with keys:
        doc_id, pid, text, type, tags, matched_paras, think
    Returns an empty list if the file is absent or FEWSHOT_JSON is None.
    """
    if not json_path or not os.path.exists(json_path):
        print("Few-shot file not found or not set – skipping.")
        return []
    with open(json_path, encoding="utf-8") as fh:
        data = json.load(fh)
    # Accept both a list directly or a dict with a 'examples' key
    if isinstance(data, dict):
        data = data.get("examples", [])
    print(f"Loaded {len(data)} few-shot examples from: {json_path}")
    return data


FEWSHOT_EXAMPLES = load_fewshot_examples(FEWSHOT_JSON if os.path.exists(str(FEWSHOT_JSON)) else None)

In [ ]:
def load_model_and_tokenizer(model_path: str):
    """Load Qwen2.5-7B with 4-bit NF4 quantization via BitsAndBytes.

    Generation is always deterministic: do_sample=False.
    Temperature / top_p / top_k are never set to avoid interference.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    print(f"Loading tokenizer from {model_path} …")
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        trust_remote_code=True,
        padding_side="left",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model from {model_path} (4-bit NF4) …")
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    print("Model loaded. Device map:", {k: str(v) for k, v in model.hf_device_map.items()
                                         if hasattr(model, 'hf_device_map')})
    return tokenizer, model


if os.path.exists(MODEL_PATH):
    tokenizer, model = load_model_and_tokenizer(MODEL_PATH)
else:
    print(f"WARNING: MODEL_PATH not found ({MODEL_PATH}). "
          "Set a valid path before running inference cells.")
    tokenizer, model = None, None

In [ ]:
# ─── System prompt ────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert annotator for United Nations resolutions written in French.
Your task is to classify and link paragraphs from a UN resolution document.

PARAGRAPH TYPES:
  - "preambular": introductory / contextual paragraphs.
    They typically begin with present-participle forms such as
    "Considérant", "Rappelant", "Reconnaissant", "Notant", "Soulignant", etc.
  - "operative": action / decision paragraphs.
    They typically begin with conjugated verbs such as
    "Prie", "Décide", "Demande", "Invite", "Encourage", "Approuve", etc.

TAGS: Use ONLY labels from the provided whitelist. Do not invent or abbreviate labels.

RELATIONS (matched_paras): Link this paragraph to at most {TOP_K_LINKS} other paragraphs.
  Choose the relation type that best describes the relationship:
  - "supporting"    : the other paragraph reinforces or corroborates this one.
  - "contradictive" : the other paragraph contradicts or opposes this one.
  - "complemental"  : the other paragraph adds new, complementary information.
  - "modifying"     : the other paragraph adjusts, amends, or qualifies this one.

THINK FIELD: Write ONE concise English sentence (max 30 words) explaining your classification.
  Template examples:
    "This paragraph recalls prior resolutions and contextualises the document, classifying it as preambular."
    "This paragraph requests the Secretary-General to submit a report, classifying it as operative."
    "Confidence is reduced because the trigger word is ambiguous."
  Do NOT write in Chinese or any language other than English.

OUTPUT: Respond with exactly one JSON object. No markdown code fences, no extra text. Schema:
{{
  "para_number"      : <int>,
  "type"             : "preambular" | "operative",
  "tags"             : [<label_string>, ...],
  "matched_paras"    : {{"<pid_str>": "<relation_type>", ...}},
  "think"            : "<one English sentence>",
  "evidence_triggers": [<trigger_word>, ...],
  "boundary_check"   : "ok" | "ambiguous" | "incomplete",
  "model_confidence" : "high" | "medium" | "low",
  "uncertainty_flag" : true | false
}}
""".format(TOP_K_LINKS=TOP_K_LINKS)


def _fewshot_block(examples: list) -> str:
    """Format few-shot examples as a numbered list in the user prompt."""
    if not examples:
        return ""
    lines = ["\n### Few-Shot Examples (for reference only, do not copy verbatim)\n"]
    for i, ex in enumerate(examples[:3], 1):   # limit to 3 to save context
        snippet = {
            "para_number"      : ex.get("pid"),
            "type"             : ex.get("type"),
            "tags"             : ex.get("tags", []),
            "matched_paras"    : ex.get("matched_paras", {}),
            "think"            : ex.get("think", ""),
            "evidence_triggers": ex.get("evidence_triggers", []),
            "boundary_check"   : ex.get("boundary_check", "ok"),
            "model_confidence" : ex.get("model_confidence", "high"),
            "uncertainty_flag" : ex.get("uncertainty_flag", False),
        }
        lines.append(f"Example {i}:")
        lines.append(f"  Paragraph text: {ex.get('text', '')[:300]}")
        lines.append(f"  Expected output: {json.dumps(snippet, ensure_ascii=False)}")
        lines.append("")
    return "\n".join(lines)


def build_messages(
    doc_id: str,
    target_para: dict,
    context_paras: list,
    whitelist: list,
    fewshot_examples: list,
) -> list:
    """Return a chat message list [system, user] for the Qwen chat template.

    Args:
        doc_id          : document identifier (string).
        target_para     : dict with keys 'pid' (int) and 'text' (str).
        context_paras   : list of dicts {'pid': int, 'text': str} for the full doc.
        whitelist       : list of valid tag label strings.
        fewshot_examples: list of human-annotated examples (may be empty).
    """
    # Build a compact context block (other paragraphs, truncated)
    ctx_lines = []
    for p in context_paras:
        if p["pid"] == target_para["pid"]:
            continue
        snippet = p["text"][:200] + ("…" if len(p["text"]) > 200 else "")
        ctx_lines.append(f"  [Para {p['pid']}] {snippet}")
    context_block = "\n".join(ctx_lines) if ctx_lines else "  (no other paragraphs)"

    whitelist_block = ", ".join(f'"{lbl}"' for lbl in whitelist) if whitelist else "(none provided)"

    fewshot_block = _fewshot_block(fewshot_examples)

    user_content = (
        f"Document ID: {doc_id}\n"
        f"\n### Target Paragraph\n"
        f"Para {target_para['pid']}: {target_para['text']}\n"
        f"\n### Other Paragraphs in Document (for cross-reference)\n"
        f"{context_block}\n"
        f"\n### Allowed Tags (whitelist)\n"
        f"{whitelist_block}\n"
        f"{fewshot_block}"
        f"\nNow annotate **Para {target_para['pid']}** following the schema above."
    )

    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
    ]

In [ ]:
# ─── Trigger word lists (UN-resolution French) ───────────────────────────────
_PREAMBULAR_TRIGGERS = [
    "affirmant", "ayant adopté", "ayant examiné", "ayant reçu",
    "conscient", "constatant", "considérant", "convaincu",
    "déplorant", "encouragé par", "entendant", "exprimant",
    "ferme conviction", "gravement préoccupé", "guidé par",
    "notant", "notant également", "notant avec satisfaction",
    "prenant acte", "prenant note", "préoccupé",
    "profondément préoccupé", "profondément convaincu",
    "rappelant", "reconnaissant", "réaffirmant",
    "se déclarant", "se félicitant", "soulignant",
    "tenant compte", "vivement préoccupé",
]

_OPERATIVE_TRIGGERS = [
    "appelle", "appelle instamment", "approuve", "autorise",
    "condamne", "confirme", "décide", "déclare", "demande",
    "demande instamment", "encourage", "exhorte", "félicite",
    "insiste", "invite", "note", "prie", "prie instamment",
    "proclame", "réaffirme", "recommande", "réitère",
    "remercie", "salue", "souligne", "souhaite", "se félicite",
]

# Sort longest-first so multi-word phrases match before their sub-phrases
_PREAMBULAR_TRIGGERS.sort(key=len, reverse=True)
_OPERATIVE_TRIGGERS.sort(key=len, reverse=True)


def extract_triggers(text: str) -> dict:
    """Return lists of matching preambular and operative trigger words found in *text*.

    Matching is case-insensitive and whole-word (simple boundary heuristic).
    """
    text_lower = text.lower()
    found_pre = []
    found_op  = []
    for phrase in _PREAMBULAR_TRIGGERS:
        # Simple substring with word-boundary check on first char
        idx = text_lower.find(phrase)
        if idx != -1:
            before_ok = (idx == 0) or not text_lower[idx - 1].isalpha()
            after_pos = idx + len(phrase)
            after_ok  = (after_pos >= len(text_lower)) or not text_lower[after_pos].isalpha()
            if before_ok and after_ok:
                found_pre.append(phrase)
    for phrase in _OPERATIVE_TRIGGERS:
        idx = text_lower.find(phrase)
        if idx != -1:
            before_ok = (idx == 0) or not text_lower[idx - 1].isalpha()
            after_pos = idx + len(phrase)
            after_ok  = (after_pos >= len(text_lower)) or not text_lower[after_pos].isalpha()
            if before_ok and after_ok:
                found_op.append(phrase)
    return {"preambular": found_pre, "operative": found_op}


# Quick smoke test
_demo = "Considérant les résolutions antérieures, le Conseil prie le Secrétaire général de soumettre un rapport."
print("Trigger demo:", extract_triggers(_demo))

In [ ]:
def boundary_check(text: str, pid: int, doc_paras: list) -> str:
    """Heuristic boundary check for a paragraph.

    Returns one of: "ok", "ambiguous", "incomplete".

    Checks:
      - Too-short paragraphs (< 8 tokens / words) → "incomplete"
      - No terminal punctuation → "incomplete"
      - Starts with a continuation pattern (lowercase word, comma, semicolon) → "ambiguous"
      - Trigger contradiction: preambular triggers + index in operative zone → "ambiguous"
    """
    text = text.strip()
    words = text.split()

    if len(words) < 8:
        return "incomplete"

    # No terminal punctuation
    if text and text[-1] not in set('.;,)"\u00bb'):
        return "incomplete"

    # Starts with lowercase (likely a continuation fragment)
    first_char = text[0] if text else ""
    if first_char.islower():
        return "ambiguous"

    # Starts with a bullet / sub-item marker
    if re.match(r"^[\-–•]\s", text) or re.match(r"^[a-z]\)", text):
        return "ambiguous"

    # Cross-check trigger type vs. paragraph position in document
    triggers = extract_triggers(text)
    operative_zone_start = len(doc_paras) // 2   # rough heuristic: 2nd half = operative
    if triggers["preambular"] and not triggers["operative"] and pid > operative_zone_start:
        return "ambiguous"
    if triggers["operative"] and not triggers["preambular"] and pid <= operative_zone_start // 2:
        return "ambiguous"

    return "ok"


# Quick smoke test
_doc = [{"pid": i, "text": f"Para {i}"} for i in range(1, 21)]
print("boundary_check demo:", boundary_check(
    "Prie le Secrétaire général de lui soumettre, à sa soixante-huitième session, "
    "un rapport sur l'application de la présente résolution.",
    pid=15, doc_paras=_doc
))

In [ ]:
_VALID_TYPES     = frozenset({"preambular", "operative"})
_VALID_RELATIONS = frozenset({"supporting", "contradictive", "complemental", "modifying"})


def check_constraints(
    pred: dict,
    whitelist: list,
    valid_pids: set,
    top_k_links: int = TOP_K_LINKS,
) -> dict:
    """Enforce schema constraints in-place on *pred*.

    Returns a diagnostics dict with keys:
      - errors   : list of blocking issues (prediction is unusable if non-empty)
      - warnings : list of corrected / soft issues
      - corrected: bool, True if any in-place correction was applied
    """
    errors   = []
    warnings = []
    corrected = False

    # ── 1. type ──────────────────────────────────────────────────────────────
    if pred.get("type") not in _VALID_TYPES:
        # Attempt auto-correction via trigger evidence
        triggers = extract_triggers(pred.get("_text", ""))
        if triggers["operative"] and not triggers["preambular"]:
            pred["type"] = "operative"
            warnings.append(f"type corrected to 'operative' via trigger evidence")
            corrected = True
        elif triggers["preambular"] and not triggers["operative"]:
            pred["type"] = "preambular"
            warnings.append(f"type corrected to 'preambular' via trigger evidence")
            corrected = True
        else:
            errors.append(f"invalid_type: {pred.get('type')!r}")

    # ── 2. tags whitelist ─────────────────────────────────────────────────────
    if whitelist:
        original_tags = pred.get("tags", [])
        if not isinstance(original_tags, list):
            original_tags = []
            pred["tags"] = []
            warnings.append("tags field was not a list – reset to []")
            corrected = True
        cleaned_tags = [t for t in original_tags if t in whitelist]
        oov_tags = [t for t in original_tags if t not in whitelist]
        if oov_tags:
            warnings.append(f"tags not in whitelist removed: {oov_tags}")
            pred["tags"] = cleaned_tags
            corrected = True

    # ── 3. matched_paras – pid-in-context + valid relations ─────────────────
    raw_matched = pred.get("matched_paras", {})
    if not isinstance(raw_matched, dict):
        pred["matched_paras"] = {}
        warnings.append("matched_paras was not a dict – reset to {}")
        corrected = True
        raw_matched = {}

    cleaned_matched = {}
    bad_pids = []
    bad_rels = []
    for pid_str, rel in raw_matched.items():
        try:
            pid_val = int(pid_str)
        except (ValueError, TypeError):
            bad_pids.append(pid_str)
            continue
        if pid_val not in valid_pids:
            bad_pids.append(pid_str)
            continue
        if rel not in _VALID_RELATIONS:
            bad_rels.append((pid_str, rel))
            rel = "supporting"   # fallback to safest relation
            corrected = True
        cleaned_matched[str(pid_val)] = rel

    if bad_pids:
        warnings.append(f"matched_paras: pids not in context removed: {bad_pids}")
        corrected = True
    if bad_rels:
        warnings.append(f"matched_paras: invalid relations replaced with 'supporting': {bad_rels}")

    # ── 4. Top-K links cap ────────────────────────────────────────────────────
    if len(cleaned_matched) > top_k_links:
        # Keep the first top_k_links entries (insertion-order stable in Python 3.7+)
        overflow = list(cleaned_matched.keys())[top_k_links:]
        cleaned_matched = dict(list(cleaned_matched.items())[:top_k_links])
        warnings.append(f"matched_paras capped to {top_k_links}; removed pids: {overflow}")
        corrected = True

    pred["matched_paras"] = cleaned_matched

    # ── 5. Optional field defaults ────────────────────────────────────────────
    pred.setdefault("think",             "Classification based on structural position and trigger words.")
    pred.setdefault("evidence_triggers", [])
    pred.setdefault("boundary_check",    "ok")
    pred.setdefault("model_confidence",  "medium")
    pred.setdefault("uncertainty_flag",  len(errors) > 0 or len(warnings) > 2)

    return {"errors": errors, "warnings": warnings, "corrected": corrected}


# Quick smoke test
_sample_pred = {
    "para_number"   : 3,
    "type"          : "operative",
    "tags"          : ["RealLabel", "FakeLabel"],
    "matched_paras" : {"1": "supporting", "99": "contradictive", "2": "unknown_rel"},
    "_text"         : "Prie le Secrétaire général.",
}
_diag = check_constraints(_sample_pred, whitelist=["RealLabel"], valid_pids={1, 2, 3, 4, 5}, top_k_links=5)
print("Diagnostics:", _diag)
print("Corrected pred:", {k: v for k, v in _sample_pred.items() if not k.startswith("_")})

In [ ]:
_REQUIRED_KEYS = {
    "para_number", "type", "tags", "matched_paras",
    "think", "evidence_triggers", "boundary_check",
    "model_confidence", "uncertainty_flag",
}


def _extract_json_from_text(text: str) -> str:
    """Extract the first JSON object from raw model output.

    Handles cases where the model wraps the object in markdown fences.
    """
    # Strip markdown fences
    text = re.sub(r"```(?:json)?", "", text).strip()
    # Find the first { … } block
    start = text.find("{")
    if start == -1:
        return text
    depth = 0
    for i, ch in enumerate(text[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    return text[start:]


def parse_response(raw_text: str, para_pid: int) -> dict:
    """Parse and lightly repair the model's JSON response.

    Returns a dict. On parse failure, returns a minimal error dict.
    """
    raw_text = raw_text.strip()
    json_str = _extract_json_from_text(raw_text)
    try:
        pred = json.loads(json_str)
    except json.JSONDecodeError as exc:
        # Try to salvage by fixing trailing commas (common LLM artefact)
        fixed = re.sub(r",\s*([}\]])", r"\1", json_str)
        try:
            pred = json.loads(fixed)
        except json.JSONDecodeError:
            return {
                "para_number"      : para_pid,
                "type"             : "preambular",
                "tags"             : [],
                "matched_paras"    : {},
                "think"            : "Parse error – could not decode model output.",
                "evidence_triggers": [],
                "boundary_check"   : "incomplete",
                "model_confidence" : "low",
                "uncertainty_flag" : True,
                "_parse_error"     : str(exc),
                "_raw_output"      : raw_text[:500],
            }

    # Ensure para_number is set correctly
    if "para_number" not in pred:
        pred["para_number"] = para_pid
    else:
        try:
            pred["para_number"] = int(pred["para_number"])
        except (TypeError, ValueError):
            pred["para_number"] = para_pid

    # Convert matched_paras keys to strings (model may emit ints)
    if isinstance(pred.get("matched_paras"), dict):
        pred["matched_paras"] = {str(k): v for k, v in pred["matched_paras"].items()}

    # Ensure tags is a list
    if not isinstance(pred.get("tags"), list):
        pred["tags"] = []

    return pred


# Quick smoke test
_raw = '{"para_number": 5, "type": "operative", "tags": ["A"], "matched_paras": {"3": "supporting"}, "think": "Operative.", "evidence_triggers": ["prie"], "boundary_check": "ok", "model_confidence": "high", "uncertainty_flag": false}'
print("Parsed:", parse_response(_raw, para_pid=5))

In [ ]:
def generate_prediction(
    doc_id: str,
    target_para: dict,
    context_paras: list,
    whitelist: list,
    fewshot_examples: list,
    tokenizer,
    model,
    max_new_tokens: int = MAX_NEW_TOKENS,
    ctx_window: int = CTX_WINDOW,
) -> dict:
    """Run the model on one paragraph and return a parsed, constraint-checked dict.

    Args:
        doc_id          : document identifier string.
        target_para     : {'pid': int, 'text': str}
        context_paras   : list of {'pid': int, 'text': str} for the full doc.
        whitelist       : list of valid tag labels.
        fewshot_examples: list of few-shot dicts (may be empty).
        tokenizer       : HuggingFace tokenizer.
        model           : HuggingFace model in eval mode.
        max_new_tokens  : max tokens to generate.
        ctx_window      : total context window size for truncation.
    """
    # Build chat messages
    messages = build_messages(
        doc_id=doc_id,
        target_para=target_para,
        context_paras=context_paras,
        whitelist=whitelist,
        fewshot_examples=fewshot_examples,
    )

    # Apply chat template
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenise and truncate to ctx_window - max_new_tokens
    max_input_tokens = ctx_window - max_new_tokens
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
    ).to(model.device)

    # Deterministic generation – do_sample=False; no temperature / top_p / top_k
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only newly generated tokens
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Parse JSON
    pred = parse_response(raw_output, para_pid=target_para["pid"])

    # Inject the source text for constraint-checker's trigger fallback
    pred["_text"] = target_para["text"]

    # ── Post-processing ───────────────────────────────────────────────────────
    # 1. Fill evidence_triggers from rule-based extractor if empty
    triggers = extract_triggers(target_para["text"])
    all_triggers = triggers["preambular"] + triggers["operative"]
    if not pred.get("evidence_triggers"):
        pred["evidence_triggers"] = all_triggers

    # 2. Boundary check
    pred["boundary_check"] = boundary_check(
        target_para["text"],
        pid=target_para["pid"],
        doc_paras=context_paras,
    )

    # 3. Constraint checks (in-place corrections)
    valid_pids = {p["pid"] for p in context_paras} - {target_para["pid"]}
    diagnostics = check_constraints(
        pred, whitelist=whitelist, valid_pids=valid_pids, top_k_links=TOP_K_LINKS
    )

    # 4. Attach metadata
    pred["_doc_id"]      = doc_id
    pred["_pid"]         = target_para["pid"]
    pred["_raw_output"]  = raw_output[:500]     # keep a snippet for debugging
    pred["_diagnostics"] = diagnostics

    return pred

In [ ]:
RAW_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "pseudo_labels_v2.jsonl")


def build_done_set(output_path: str) -> set:
    """Return the set of (doc_id, pid) tuples already written to *output_path*."""
    done = set()
    if not os.path.exists(output_path):
        return done
    with open(output_path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                done.add((str(rec["_doc_id"]), int(rec["_pid"])))
            except (json.JSONDecodeError, KeyError):
                pass
    print(f"Resume: found {len(done)} already-completed (doc_id, pid) pairs in {output_path}")
    return done


def run_inference_loop(
    coreset: pd.DataFrame,
    whitelist: list,
    fewshot_examples: list,
    tokenizer,
    model,
    output_path: str = RAW_OUTPUT_PATH,
) -> list:
    """Main inference loop.

    Processes every paragraph in *coreset*, skipping pairs already present in
    *output_path* (resume logic).  Writes each result immediately to the JSONL
    file so progress is not lost on interruption.

    Returns the full list of result dicts (including previously completed ones).
    """
    assert tokenizer is not None and model is not None, (
        "Model and tokenizer must be loaded before calling run_inference_loop."
    )

    done_set   = build_done_set(output_path)
    results    = []

    # Pre-load existing results
    if os.path.exists(output_path):
        with open(output_path, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    try:
                        results.append(json.loads(line))
                    except json.JSONDecodeError:
                        pass

    # Group by doc_id so we can pass full-document context
    doc_groups = coreset.groupby("doc_id", sort=False)

    new_count = 0
    with open(output_path, "a", encoding="utf-8") as out_fh:
        for doc_id, doc_df in tqdm(doc_groups, desc="Documents"):
            doc_id = str(doc_id)
            doc_paras = [
                {"pid": int(row.pid), "text": str(row.text)}
                for row in doc_df.sort_values("pid").itertuples()
            ]

            for target_para in tqdm(doc_paras, desc=f"  {doc_id}", leave=False):
                key = (doc_id, target_para["pid"])
                if key in done_set:
                    continue    # ← resume: skip already-done rows

                pred = generate_prediction(
                    doc_id=doc_id,
                    target_para=target_para,
                    context_paras=doc_paras,
                    whitelist=whitelist,
                    fewshot_examples=fewshot_examples,
                    tokenizer=tokenizer,
                    model=model,
                )

                results.append(pred)
                done_set.add(key)
                new_count += 1

                # Write immediately (buffered, flush every 10 rows)
                out_fh.write(json.dumps(pred, ensure_ascii=False) + "\n")
                if new_count % 10 == 0:
                    out_fh.flush()

    print(f"\nInference complete: {new_count} new rows written (total: {len(results)}).")
    return results


# ── Run ────────────────────────────────────────────────────────────────────────
if model is not None and len(coreset_df) > 0:
    all_results = run_inference_loop(
        coreset=coreset_df,
        whitelist=WHITELIST,
        fewshot_examples=FEWSHOT_EXAMPLES,
        tokenizer=tokenizer,
        model=model,
        output_path=RAW_OUTPUT_PATH,
    )
else:
    print("Skipping inference: model not loaded or coreset is empty.")
    # Load any existing results for downstream cells
    all_results = []
    if os.path.exists(RAW_OUTPUT_PATH):
        with open(RAW_OUTPUT_PATH, encoding="utf-8") as fh:
            all_results = [json.loads(l) for l in fh if l.strip()]
    print(f"Loaded {len(all_results)} existing results from {RAW_OUTPUT_PATH}")

In [ ]:
CLEAN_OUTPUT_PATH  = os.path.join(OUTPUT_DIR, "pseudo_labels_v2_clean.jsonl")
REVIEW_QUEUE_PATH  = os.path.join(OUTPUT_DIR, "review_queue_v2.jsonl")

# Fields kept in the clean output (internal / debug fields are stripped)
_CLEAN_FIELDS = [
    "para_number", "type", "tags", "matched_paras",
    "think", "evidence_triggers", "boundary_check",
    "model_confidence", "uncertainty_flag",
    "_doc_id", "_pid",
]

# Criteria for sending a row to the review queue
def _needs_review(pred: dict) -> bool:
    diag = pred.get("_diagnostics", {})
    return any([
        bool(diag.get("errors")),
        pred.get("boundary_check") in ("ambiguous", "incomplete"),
        pred.get("model_confidence") == "low",
        pred.get("uncertainty_flag") is True,
        bool(pred.get("_parse_error")),
    ])


clean_records  = []
review_records = []

for pred in all_results:
    clean = {k: pred[k] for k in _CLEAN_FIELDS if k in pred}
    clean_records.append(clean)
    if _needs_review(pred):
        review_records.append(pred)

with open(CLEAN_OUTPUT_PATH, "w", encoding="utf-8") as fh:
    for rec in clean_records:
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

with open(REVIEW_QUEUE_PATH, "w", encoding="utf-8") as fh:
    for rec in review_records:
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Clean records   : {len(clean_records):>5}  → {CLEAN_OUTPUT_PATH}")
print(f"Review-queue    : {len(review_records):>5}  → {REVIEW_QUEUE_PATH}")

In [ ]:
QUALITY_REPORT_PATH = os.path.join(OUTPUT_DIR, "quality_report_v2.json")

def build_quality_report(results: list, review_records: list, whitelist: list) -> dict:
    """Aggregate quality statistics over all inference results."""
    total = len(results)
    if total == 0:
        return {"total": 0, "note": "No results to report."}

    type_counts    = collections.Counter(r.get("type") for r in results)
    bc_counts      = collections.Counter(r.get("boundary_check") for r in results)
    conf_counts    = collections.Counter(r.get("model_confidence") for r in results)
    parse_errors   = sum(1 for r in results if r.get("_parse_error"))
    constraint_warns = sum(
        1 for r in results
        if r.get("_diagnostics", {}).get("warnings")
    )
    constraint_errs = sum(
        1 for r in results
        if r.get("_diagnostics", {}).get("errors")
    )
    tags_used = collections.Counter(
        tag for r in results for tag in r.get("tags", [])
    )
    docs = len({r.get("_doc_id") for r in results})

    report = {
        "generated_at"     : datetime.datetime.utcnow().isoformat() + "Z",
        "total_paragraphs" : total,
        "total_documents"  : docs,
        "type_distribution": dict(type_counts),
        "boundary_check"   : dict(bc_counts),
        "model_confidence" : dict(conf_counts),
        "parse_errors"     : parse_errors,
        "constraint_warnings"  : constraint_warns,
        "constraint_errors"    : constraint_errs,
        "review_queue_size"    : len(review_records),
        "top_20_tags_used"     : dict(tags_used.most_common(20)),
        "whitelist_size"       : len(whitelist),
        "tags_not_in_whitelist": sorted(
            {t for r in results for t in r.get("tags", []) if t not in whitelist}
        )[:20],
    }
    return report


quality_report = build_quality_report(all_results, review_records, WHITELIST)

with open(QUALITY_REPORT_PATH, "w", encoding="utf-8") as fh:
    json.dump(quality_report, fh, ensure_ascii=False, indent=2)

print(f"Quality report saved: {QUALITY_REPORT_PATH}")
print(json.dumps(quality_report, indent=2, ensure_ascii=False))

In [ ]:
# ─── Submission schema ────────────────────────────────────────────────────────
# {
#   "TEXT_ID": <string>,
#   "METADATA": {
#     "structure": {
#       "preambular_paras": [<int>, ...],
#       "operative_paras" : [<int>, ...],
#       "think"           : "<string>"
#     }
#   },
#   "body": {
#     "paras": [
#       {
#         "pid"          : <int>,
#         "type"         : "preambular" | "operative",
#         "tags"         : [<string>, ...],
#         "matched_paras": {"<pid_str>": "<relation>", ...},
#         "think"        : "<string>"
#       },
#       ...
#     ]
#   }
# }

_SUBMISSION_PARA_FIELDS = ("type", "tags", "matched_paras", "think")


def _make_structure_think(preambular_pids: list, operative_pids: list) -> str:
    """Generate a short templated think string for the document-level structure."""
    n_pre = len(preambular_pids)
    n_op  = len(operative_pids)
    return (
        f"Document contains {n_pre} preambular paragraph(s) "
        f"and {n_op} operative paragraph(s)."
    )


def build_submission_doc(doc_id: str, doc_preds: list) -> dict:
    """Convert a list of paragraph predictions into the competition submission format.

    Args:
        doc_id    : TEXT_ID string.
        doc_preds : list of clean prediction dicts, each with at minimum:
                    _pid, type, tags, matched_paras, think.

    Returns the submission dict.
    """
    # Sort by pid
    doc_preds = sorted(doc_preds, key=lambda r: int(r.get("_pid", r.get("para_number", 0))))

    preambular_pids = [int(r["_pid"]) for r in doc_preds if r.get("type") == "preambular"]
    operative_pids  = [int(r["_pid"]) for r in doc_preds if r.get("type") == "operative"]

    paras_out = []
    for r in doc_preds:
        para_out = {
            "pid"          : int(r.get("_pid", r.get("para_number", 0))),
            "type"         : r.get("type", "preambular"),
            "tags"         : r.get("tags", []),
            "matched_paras": r.get("matched_paras", {}),
            "think"        : r.get("think", ""),
        }
        paras_out.append(para_out)

    return {
        "TEXT_ID" : doc_id,
        "METADATA": {
            "structure": {
                "preambular_paras": preambular_pids,
                "operative_paras" : operative_pids,
                "think"           : _make_structure_think(preambular_pids, operative_pids),
            }
        },
        "body": {
            "paras": paras_out,
        },
    }


def export_submissions(
    clean_records: list,
    submission_dir: str,
) -> list:
    """Write one JSON file per document to *submission_dir*.

    Returns the list of written file paths.
    """
    # Group by doc_id
    by_doc = collections.defaultdict(list)
    for rec in clean_records:
        doc_id = str(rec.get("_doc_id", "UNKNOWN"))
        by_doc[doc_id].append(rec)

    written = []
    for doc_id, preds in tqdm(by_doc.items(), desc="Exporting submissions"):
        sub_doc  = build_submission_doc(doc_id, preds)
        out_path = os.path.join(submission_dir, f"{doc_id}.json")
        with open(out_path, "w", encoding="utf-8") as fh:
            json.dump(sub_doc, fh, ensure_ascii=False, indent=2)
        written.append(out_path)

    return written


if clean_records:
    written_files = export_submissions(clean_records, SUBMISSION_DIR)
    print(f"\nExported {len(written_files)} submission file(s) to: {SUBMISSION_DIR}")
    # Show one example
    if written_files:
        with open(written_files[0], encoding="utf-8") as fh:
            sample = json.load(fh)
        print("\nSample (first document):")
        # Print only the first paragraph to keep output concise
        preview = {
            "TEXT_ID" : sample["TEXT_ID"],
            "METADATA": sample["METADATA"],
            "body"    : {"paras": sample["body"]["paras"][:2], "...": "..."},
        }
        print(json.dumps(preview, ensure_ascii=False, indent=2))
else:
    print("No clean records to export.  Run inference first.")

## Output Summary

| File | Description |
|------|-------------|
| `outputs_v2/pseudo_labels_v2.jsonl` | Raw model predictions with all internal fields |
| `outputs_v2/pseudo_labels_v2_clean.jsonl` | Submission-ready fields only |
| `outputs_v2/review_queue_v2.jsonl` | Rows flagged for human review |
| `outputs_v2/quality_report_v2.json` | Aggregate statistics (types, tags, errors) |
| `outputs_v2/submission_export/<TEXT_ID>.json` | One file per document in competition schema |

## How to resume an interrupted run

Re-run **Cell 14** (the inference loop cell).  
It reads the existing `pseudo_labels_v2.jsonl`, builds the set of already-completed `(doc_id, pid)` pairs, and skips them, processing only the remaining rows.

## Relation-type cheat-sheet

| Type | Meaning |
|------|---------|
| `supporting` | Other paragraph reinforces / corroborates this one |
| `contradictive` | Other paragraph contradicts or opposes this one |
| `complemental` | Other paragraph adds **new** complementary information |
| `modifying` | Other paragraph **adjusts, amends, or qualifies** this one |

> **Complemental vs. Modifying**: *complemental* introduces additional context; *modifying* changes or qualifies the content of the linked paragraph.